# PE5 (ESCC): grouping, protein adjustment and differential analysis

Inputs
- `PE5_profile.xlsx`: case ID, substage and Firmiana experiment ID of every proteome / phospho sample
- `PE5_phosphoproteome_data.xlsx`, `PE5_proteome_data.xlsx`: intensities

Groups come from the tissue substage: 1 -> T0, 2-7 -> T1, T2 (8) -> T2, T3 (9) -> T3.
Comparisons: T1, T2, T3 and T2+T3, each against T0.

Each comparison is run as protein, phospho, phospho adjusted for protein (Method A) and Method B.

## Setup

In [ ]:
import re
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import phosphokit as pk

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
DATA = Path("../../data/raw/PE5")
PROFILE_FILE = DATA / "PE5_profile.xlsx"
PHOSPHO_FILE = DATA / "PE5_phosphoproteome_data.xlsx"
PROTEIN_FILE = DATA / "PE5_proteome_data.xlsx"
PROTEIN_SHEET = "Sheet1"          # the 786 proteome samples listed in the profile
DATASET = "PE5"
OUT_DIR = Path(f"../../results/{DATASET}")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

S = pk.Settings(normalization="quantile_common", norm_common_frac=0.7,
                missing_method="none", min_per_group=3)

GROUP_ORDER = ["T0", "T1", "T2", "T3"]
GROUP_COLORS = {"T0": "#55A868", "T1": "#DD8452", "T2": "#C44E52", "T3": "#8172B3"}
COMPARISONS = {"T1_vs_T0": (["T1"], ["T0"]), "T2_vs_T0": (["T2"], ["T0"]),
               "T3_vs_T0": (["T3"], ["T0"]), "T2T3_vs_T0": (["T2", "T3"], ["T0"])}
PROTEIN_MISSING_VALUE = 1e-5      # "not detected" placeholder in the proteome file
ZERO_IS_MISSING = True
COLLAPSE_NON_PHOSPHO_MODS = True  # merge forms that differ only in Met oxidation etc.
DUPLICATE_RULE = "sum"


def substage_to_group(substage):
    major = str(substage).split("-")[0].strip().upper()
    if major == "1":
        return "T0"
    if major in {"2", "3", "4", "5", "6", "7"}:
        return "T1"
    if major in {"8", "T2"}:
        return "T2"
    if major in {"9", "T3"}:
        return "T3"
    return None


def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()

## 1. Pairing and grouping

Samples are matched on **(case, substage)**, not on case alone: one case contributes several
substages, and only the same tissue piece may be paired.

In [ ]:
profile = {
    "proteome": pk.io.read_pe5_profile(PROFILE_FILE, "proteome_profile", "Case ID",
                                       "Substages (Experiment ID)_proteome", "proteome", substage_to_group),
    "phospho": pk.io.read_pe5_profile(PROFILE_FILE, "phospho_profile", "No.",
                                      "Substages (Experiment ID)-phosphoproteome", "phospho", substage_to_group),
}
prot_samples = profile["proteome"]
samples = pk.io.pair_by_substage(profile["phospho"], prot_samples)
print(f"phospho samples {len(samples)}, with a proteome partner {samples['paired'].sum()}")
print(samples.groupby("Group").agg(samples=("Case", "size"), patients=("Case", "nunique")).reindex(GROUP_ORDER))

t0_cases = set(samples.loc[samples["Group"] == "T0", "Case"])
for name, (test, ref) in COMPARISONS.items():
    both = set(samples.loc[samples["Group"].isin(test), "Case"]) & t0_cases
    print(f"{name:11s}: {len(both)} patients in both arms")

## 2. Phosphopeptide and protein matrices

Lower-case residues mark modifications; only `s/t/y` are phosphosites, so the other lower-case
letters are folded back and the rows that become identical are merged.

In [ ]:
ph_raw = pd.read_excel(PHOSPHO_FILE, dtype={"Symbol": str})
ph_raw.columns = ["Seq", "Symbol"] + [pk.io.norm_experiment_id(c) for c in ph_raw.columns[2:]]
sample_cols = [c for c in ph_raw.columns[2:] if c in samples.index]
if ZERO_IS_MISSING:
    ph_raw[sample_cols] = pk.zero_to_nan(ph_raw[sample_cols])

ph_raw["Symbol"] = ph_raw["Symbol"].map(pk.fix_excel_dates)
ph_raw["Peptide"] = ph_raw["Seq"].map(pk.phospho_form) if COLLAPSE_NON_PHOSPHO_MODS else ph_raw["Seq"]
ph_raw["Feature"] = ph_raw["Symbol"] + "|" + ph_raw["Peptide"]
ph_lin = pk.collapse_duplicates(ph_raw, "Feature", sample_cols, DUPLICATE_RULE)

feat_info = ph_raw.drop_duplicates("Feature").set_index("Feature")[["Symbol", "Peptide"]].loc[ph_lin.index]
feat_info["n_phosphosites"] = feat_info["Peptide"].map(pk.prep.count_phosphosites)
print(f"{len(ph_raw)} rows -> {len(ph_lin)} phosphopeptides")

pr_raw = pd.read_excel(PROTEIN_FILE, sheet_name=PROTEIN_SHEET, dtype={"Symbol": str})
pr_raw.columns = ["Symbol"] + [pk.io.norm_experiment_id(c) for c in pr_raw.columns[1:]]
pr_raw["Symbol"] = pr_raw["Symbol"].map(pk.fix_excel_dates)
prot_cols = [c for c in pr_raw.columns[1:] if c in set(prot_samples["ExpID_proteome"])]
pr_raw[prot_cols] = pk.zero_to_nan(pr_raw[prot_cols], PROTEIN_MISSING_VALUE)
# repairing the Excel dates can produce two rows with the same symbol; here they are measured in
# complementary samples, so they are merged
prot_lin = pr_raw.groupby("Symbol")[prot_cols].mean()
print(f"proteins {prot_lin.shape[0]}, proteome samples {prot_lin.shape[1]}")

## 3. Normalization

Tumor samples detect far more features than normal samples (about 2,400 proteins in T0 against
4,600 in T2), and the extra ones are mostly low-abundance. Aligning all observed values therefore
pushes the tumor groups upward, which is why the common-feature variants exist.

In [ ]:
ph_log_raw, prot_log_raw = np.log2(ph_lin), np.log2(prot_lin)
pair_map = samples["ExpID_proteome"].dropna()
prot_in_ph_space = prot_log_raw[pair_map.values].set_axis(pair_map.index, axis=1)

bias = pd.concat({"phospho": pk.normalization_bias(ph_log_raw, samples["Group"], "T0", S.norm_common_frac),
                  "protein": pk.normalization_bias(prot_in_ph_space,
                                                   samples.loc[prot_in_ph_space.columns, "Group"],
                                                   "T0", S.norm_common_frac)},
                 axis=1)
print(bias.round(2))

ph_log = pk.normalize(ph_log_raw, S.normalization, S.norm_common_frac)
prot_log = pk.normalize(prot_log_raw, S.normalization, S.norm_common_frac)
ph_log, _ = pk.impute(ph_log, S.missing_method, S.impute_min_valid_frac, S)
prot_log, _ = pk.impute(prot_log, S.missing_method, S.impute_min_valid_frac, S)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
pk.sample_boxplot(ph_log_raw, axes[0], samples["Group"], GROUP_COLORS, "Phospho, raw")
pk.sample_boxplot(ph_log, axes[1], samples["Group"], GROUP_COLORS, f"Phospho, {S.normalization}")
savefig("01_boxplots")

## 4. Protein adjustment input

In [ ]:
# protein values laid out in the phospho sample space (NaN for the sample without a partner)
protein_by_sample = pd.DataFrame(np.nan, index=prot_log.index, columns=ph_log.columns)
protein_by_sample[pair_map.index] = prot_log[pair_map.values].values

ph_adj = pk.adjusted_matrix(ph_log, protein_by_sample, feat_info["Symbol"])
protein_by_feature = protein_by_sample.loc[feat_info.loc[ph_adj.index, "Symbol"]].set_axis(ph_adj.index)
print(f"adjustable peptides: {len(ph_adj)} / {len(feat_info)} ({len(ph_adj) / len(feat_info):.1%})")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
site_r = pk.phospho_protein_correlation(ph_log.loc[ph_adj.index], protein_by_feature, ax)
savefig("02_phospho_protein_correlation")

## 5. Sample structure

Only tumor-vs-normal *ratios* would make a T-vs-N plot impossible, but here the values are
intensities, so the samples can be embedded directly. Gaps are filled for the plot only.

In [ ]:
fig, _ = pk.embedding_plot(ph_log, samples["Group"], GROUP_COLORS, methods=("PCA", "UMAP", "t-SNE"),
                           min_valid=0.7, fill="downshift", seed=S.seed, title_prefix="phospho ")
savefig("03_phospho_pca_umap_tsne")

## 6. Tests

Samples of one patient within one group are averaged (several T1 lesions of a patient are not
independent replicates of T1). Patients present in both arms - only T1 vs T0 here - are blocked
with `duplicateCorrelation`.

In [ ]:
res = {k: {} for k in ["protein", "phospho", "adjA", "adjB"]}
info = {k: {} for k in res}

for name, (test, ref) in COMPARISONS.items():
    res["phospho"][name], info["phospho"][name] = pk.two_group_test(ph_log, samples, test, ref, S)
    res["protein"][name], info["protein"][name] = pk.two_group_test(protein_by_sample, samples, test, ref, S)
    res["adjA"][name], info["adjA"][name] = pk.two_group_test(ph_adj, samples, test, ref, S)
    res["adjB"][name] = pk.method_b(res["phospho"][name], res["protein"][name], feat_info["Symbol"])

overview = pd.DataFrame([{"analysis": k, "comparison": name, "tested": len(r),
                          "Up": (pk.direction(r, S) == "Up").sum(),
                          "Down": (pk.direction(r, S) == "Down").sum(),
                          "rho": info[k].get(name, {}).get("rho"),
                          "prior_d0": info[k].get(name, {}).get("d0")}
                         for k in res for name, r in res[k].items()])
overview

## 7. Figures

In [ ]:
label = (feat_info["Symbol"].astype(str)).to_dict()
analyses = ["protein", "phospho", "adjA", "adjB"]

fig, axes = plt.subplots(len(analyses), len(COMPARISONS), figsize=(4.3 * len(COMPARISONS), 3.8 * len(analyses)))
for i, k in enumerate(analyses):
    for j, name in enumerate(COMPARISONS):
        pk.volcano(res[k][name], axes[i, j], f"{k} | {name}",
                   labels=None if k == "protein" else label, n_label=6, settings=S)
savefig("04_volcano_all")

for name in COMPARISONS:
    pk.save_volcanoes({f"{k}_{name}": res[k][name] for k in ["protein", "phospho", "adjA"]},
                      FIG_DIR / "volcano_individual", labels=label, settings=S)

pk.pvalue_histograms({f"{k} {n}": res[k][n] for k in analyses for n in ["T2T3_vs_T0"]})
savefig("05_pvalue_histograms")

fig, axes = plt.subplots(len(COMPARISONS), 2, figsize=(10, 3.4 * len(COMPARISONS)))
for i, name in enumerate(COMPARISONS):
    pk.method_comparison(res["adjA"][name], res["adjB"][name], axes[i, 0], axes[i, 1], site_correlation=site_r)
    axes[i, 0].set_ylabel(f"{name}\nMethod B")
savefig("06_methodA_vs_B")

## 8. Categories and tables

The unadjusted and adjusted lists are both exported: they answer different questions, and peptides
whose protein was not quantified exist only in the unadjusted one.

In [ ]:
categories = {name: pk.category_table(res["protein"][name], res["phospho"][name], res["adjA"][name],
                                      feat_info["Symbol"], S) for name in COMPARISONS}
cat_counts = pd.DataFrame({n: c["category"].value_counts() for n, c in categories.items()}) \
    .reindex(pk.CATEGORY_ORDER).fillna(0).astype(int)
print(cat_counts)

fig, ax = plt.subplots(figsize=(11, 4))
plot_tab = cat_counts.drop(index="Not significant")
bottom = np.zeros(len(COMPARISONS))
for cat in plot_tab.index:
    ax.bar(list(COMPARISONS), plot_tab.loc[cat], bottom=bottom, color=pk.CATEGORY_COLORS[cat], label=cat)
    bottom += plot_tab.loc[cat].values
ax.set_ylabel("# phosphopeptides")
ax.set_title("Significant phosphopeptides by category")
ax.legend(fontsize=8)
savefig("07_site_categories")

In [ ]:
def pick(result, prefix, cols=("logFC", "P", "adj.P")):
    out = result[list(cols)].add_prefix(f"{prefix}_")
    out[f"{prefix}_direction"] = pk.direction(result, S)
    return out


annot = feat_info[["Symbol", "Peptide", "n_phosphosites"]]
out_file = OUT_DIR / f"{DATASET}_phospho_results.xlsx"
with pd.ExcelWriter(out_file) as xw:
    samples.reset_index().to_excel(xw, sheet_name="Sample_groups", index=False)
    overview.to_excel(xw, sheet_name="Overview", index=False)
    for name in COMPARISONS:
        table = annot.loc[res["adjA"][name].index].join(pick(res["phospho"][name], "Phospho"))
        table = table.join(pick(res["protein"][name], "Protein", cols=("logFC", "P")), on="Symbol")
        table = table.join(pick(res["adjA"][name], "Adjusted"))
        table["Adjusted_MethodB_direction"] = pk.direction(res["adjB"][name], S).reindex(table.index)
        table["Category"] = categories[name]["category"]
        table.sort_values("Adjusted_P").to_excel(xw, sheet_name=f"Adjusted_{name}"[:31])

        dep = annot.join(pick(res["phospho"][name], "Phospho"), how="inner")
        dep = dep[dep["Phospho_direction"] != ""].sort_values("Phospho_P")
        dep["Protein_quantified"] = dep.index.isin(ph_adj.index)
        dep.to_excel(xw, sheet_name=f"PhosphoDEP_{name}"[:31])
    cat_counts.to_excel(xw, sheet_name="Categories")
print("saved:", out_file)